# AIML428 - Assignment1, Step 6 Bert

In [32]:
import numpy.dtypes
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification


In [33]:
def tableHeader():
    doc = "|Algorithm| label | accuracy | precision | recall | f1 |\n"
    doc += "| --- | --- | --- | --- |--- | --- |\n"
    return doc

def tableScore(labels, doc, prefix=None, y_true=None, y_prediction=None):
    rows = ""
    accuracy = accuracy_score(y_true, y_prediction)
    precisions = precision_score(y_true, y_prediction, labels=labels, average=None)
    recalls = recall_score(y_true, y_prediction, labels=labels, average=None)
    f1s = f1_score(y_true, y_prediction, labels=labels, average=None)
    for label, precision, recall, f1 in zip(labels, precisions, recalls, f1s):
        rows += f"| {prefix} | {label} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        prefix = " "
    return doc + rows

def tableFooter(doc):
    display(Markdown(doc))

def report(string):
    display(Markdown(string))

enable_debug = True

def debug(string):
    if (enable_debug):
        print(string)

## Try to use a GPU for go faster juice

In [34]:
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Metal (GPU) is available")
# else:
#     device = torch.device("cpu")
#     print("GPU not available, using CPU")
#
# # Encourage Keras to use Torch, and to use the GPU.
# os.environ["KERAS_BACKEND"] = "torch"
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

## Load the data.

In [35]:
df_csv = pd.read_csv('bbc_converted.csv')

feature_columns = ['text']
target_column = 'category'

X_csv = df_csv[feature_columns]
Y_csv = df_csv[target_column]
# unique converts to an NDArray, so the sort needs to happen first.
labels = Y_csv.sort_values().unique()
print(labels)
num_labels = len(labels)
print(num_labels)


['business' 'entertainment' 'politics' 'sport' 'tech']
5


In [36]:
# https://xkcd.com/221/ - 4 is overused
random_seed = 221

In [37]:
# Data already loaded
# X_csv is the CSV text data
# Y_csv is the CSV labels - the sorted label set.
debug(X_csv)
debug(Y_csv)
debug(labels)

                                                   text
0     musicians to tackle us red tape musicians grou...
1     u2 s desire to be number one u2 who have won t...
2     rocker doherty in on stage fight rock singer p...
3     snicket tops us box office chart the film adap...
4     ocean s twelve raids box office ocean s twelve...
...                                                 ...
2220  warning over windows word files writing a micr...
2221  fast lifts rise into record books two high spe...
2222  nintendo adds media playing to ds nintendo is ...
2223  fast moving phone viruses appear security firm...
2224  hacker threat to apple s itunes users of apple...

[2225 rows x 1 columns]
0       entertainment
1       entertainment
2       entertainment
3       entertainment
4       entertainment
            ...      
2220             tech
2221             tech
2222             tech
2223             tech
2224             tech
Name: category, Length: 2225, dtype: object
['business' 'ente

## Test/Train Split

In [38]:
X_train, X_test, Y_train, Y_test = train_test_split(X_csv, Y_csv, test_size= 0.3, random_state=random_seed)

debug(f"X columns: {X_train.columns}")
debug(f"X.head: {X_train.text.head(1)}")
debug(f"Y.head: {Y_train.head(1)}")
#debug(f"result: {X_train[0]}")

X columns: Index(['text'], dtype='object')
X.head: 1187    edwards tips idowu for euro gold world outdoor...
Name: text, dtype: object
Y.head: 1187    sport
Name: category, dtype: object


## Encode the label

Train on all the _training_ values, but apply it to the _train/_validation variables, and the _test variables.

The type of the encoding is important to BERT. OrdinalEncoder is appropriate, however it is also important to use int64 as the encoding type. Otherwise it chooses the wrong loss function and assumes a multi-label classification problem instead of a single-label classification problem.

In [39]:
# label encode the target variable
encoder = preprocessing.OrdinalEncoder(dtype=numpy.int64)

encoder.fit(Y_train.values.reshape(-1, 1))
Y_train_enc = encoder.transform(Y_train.values.reshape(-1, 1))
Y_test_enc = encoder.transform(Y_test.values.reshape(-1, 1))

print(labels)
encoder.transform(labels.reshape(-1, 1))

['business' 'entertainment' 'politics' 'sport' 'tech']


array([[0],
       [1],
       [2],
       [3],
       [4]])

# Prepare the Dataset

In [40]:
print(f"{X_train['text'].tolist()[0]}")


edwards tips idowu for euro gold world outdoor triple jump record holder and bbc pundit jonathan edwards believes phillips idowu can take gold at the european indoor championships idowu landed 17 30m at the british trials in sheffield last month to lead the world triple jump rankings it s all down to him but if he jumps as well as he did in sheffield he could win the gold medal said edwards his ability is undoubted but all his best performances seem to happen in domestic meetings idowu made his breakthrough five years ago but so far has only a commonwealth silver medal to his name edwards himself kept idowu off top spot at the manchester games but he believes the european indoors in madrid represent a chance for the 26 year old to prove his credentials as britain s top triple jumper he has to start producing at international level and here is the beginning said edwards phillips still needs to be much more consistent i m sure a victory in madrid will build up his confidence and self bel

In [41]:
tokenizer = BertTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')
encoded_data = tokenizer(X_train['text'].tolist(), padding=True, truncation=True, return_tensors='pt', do_lower_case=True)
input_ids = encoded_data['input_ids']
attention_mask = encoded_data['attention_mask']
labels_tensor = torch.tensor(Y_train_enc)

test_encoded_data = tokenizer(X_test['text'].tolist(), padding=True, truncation=True, return_tensors='pt', do_lower_case=True)
test_input_ids = test_encoded_data['input_ids']
test_attention_mask = test_encoded_data['attention_mask']
test_labels = torch.tensor(Y_test_enc)

test_labels


tensor([[4],
        [0],
        [2],
        [2],
        [3],
        [1],
        [4],
        [3],
        [3],
        [3],
        [1],
        [1],
        [1],
        [4],
        [2],
        [2],
        [1],
        [0],
        [4],
        [1],
        [1],
        [4],
        [2],
        [4],
        [2],
        [3],
        [0],
        [1],
        [4],
        [4],
        [0],
        [2],
        [3],
        [0],
        [3],
        [4],
        [1],
        [1],
        [1],
        [0],
        [4],
        [2],
        [1],
        [2],
        [4],
        [1],
        [3],
        [3],
        [0],
        [2],
        [4],
        [2],
        [1],
        [3],
        [4],
        [4],
        [2],
        [1],
        [3],
        [2],
        [1],
        [4],
        [2],
        [4],
        [0],
        [3],
        [0],
        [0],
        [1],
        [0],
        [0],
        [2],
        [0],
        [3],
        [3],
        [2],
        [0],

## Train Validation Split

skipping for now.

In [42]:


# validation_size=0.2
batch_size=32

# Don't shuffle, everything has to stay together
# train_input_ids, validation_input_ids, train_attention_mask, validation_attention_mask, train_labels, validation_labels = (
#     train_test_split(input_ids, attention_mask, labels_tensor, test_size=validation_size,shuffle=False))

train_input_ids = input_ids
train_attention_mask = attention_mask
train_labels = labels_tensor


## Train Bert

In [43]:
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size) # Small batch size for small dataset

test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [50]:

model = BertForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english', num_labels=num_labels, ignore_mismatched_sizes=True)


You are using a model of type `distilbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                                                                      | Status     |                                                                                       
-------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin1.bias            | UNEXPECTED |                                                                                       
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.sa_layer_norm.bias       | UNEXPECTED |                                                                                       
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.weight | UNEXPECTED |                                                                                       
distilbert.transformer.layer.

In [51]:
# 5. Set up optimizer and loss_function
optimizer = AdamW(model.parameters(), lr=2e-5)

In [52]:
num_epochs=5

history = {"average_training_loss": []}
for epoch in range(num_epochs): # Train for a few epochs
    model.train() # Set model to training mode
    training_loss = 0
    for batch in train_dataloader:
        batch_input_ids, batch_attention_mask, batch_labels = batch

        model.zero_grad() # Clear previous gradients

        outputs = model(batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)

        loss = outputs.loss
        loss.backward() # Backpropagate
        optimizer.step() # Update weights
        training_loss += loss.item()

    average_training_loss = training_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}, Average Training Loss: {average_training_loss}")
    history["average_training_loss"].append(average_training_loss)

history

Epoch 1, Average Training Loss: 1.6449313066443618
Epoch 2, Average Training Loss: 1.6153076887130737
Epoch 3, Average Training Loss: 1.4393464862083902
Epoch 4, Average Training Loss: 1.0142233760989443
Epoch 5, Average Training Loss: 0.6327878571286494


{'average_training_loss': [1.6449313066443618,
  1.6153076887130737,
  1.4393464862083902,
  1.0142233760989443,
  0.6327878571286494]}

In [53]:
model.eval() # Set model to evaluation mode

# Once the model is trained, then bfi this bastard.

def count_correct(logits, values):
    predicted_token_ids = torch.argmax(logits, dim=-1)
    flat_pred = predicted_token_ids.flatten()
    flat_values = values.flatten()
    merged = list(zip(flat_pred, flat_values))
    accuracy = 0
    for a in merged:
        pred, val = a
        if torch.equal(pred, val):
            accuracy += 1
    return accuracy

correct = 0
for batch in test_dataloader:
    batch_input_ids, batch_attention_mask, batch_labels = batch
    with torch.no_grad():
        outputs = model(batch_input_ids, attention_mask=batch_attention_mask)
        logits = outputs.logits
        correct += count_correct(logits, batch_labels)

accuracy = correct / Y_test.shape[0]
report(f"*Accuracy:* {accuracy}")


*Accuracy:* 0.6766467065868264